# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata (not as a dictionary)
metadata = dataset.metadata

# Print dataset overview
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Authors (@id): {getattr(metadata, 'author', [])}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review the available record sets and fields, referencing entities by their `@id` values.

In [ ]:
record_sets = getattr(metadata, 'recordSet', [])
print(f"Found {len(record_sets)} record sets.")

# Print details for each record set (@id)
for rset in record_sets:
    rset_id = getattr(rset, '@id', None)
    print(f"Record set @id: {rset_id}")
    fields = getattr(rset, 'field', [])
    print(f" Fields:")
    for fld in fields:
        fld_id = getattr(fld, '@id', None)
        fld_name = getattr(fld, 'name', '')
        fld_dtype = getattr(fld, 'dataType', 'unknown')
        print(f"  • {fld_id} | name: {fld_name} | dataType: {fld_dtype}")
    print("")

# Example: Print first few records from a record set
if record_sets:
    first_rset = getattr(record_sets[0], '@id', None)
    print(f"Sample records from record set @id: {first_rset}")
    for rec in dataset.records(record_set=first_rset):
        print(rec)
        break  # Print only the first sample

## 3. Data Extraction
Load data from all record sets (referenced by their `@id`) into DataFrames. Data columns also referenced by their `@id`.

In [ ]:
dataframes = {}
record_set_ids = []

# Collect all record set @id values
for rset in getattr(metadata, 'recordSet', []):
    rset_id = getattr(rset, '@id', None)
    if rset_id:
        record_set_ids.append(rset_id)

# Load each record set as a DataFrame
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"Loaded record set @id: {rid} (shape={df.shape})")

# Print columns from the first record set
if record_set_ids:
    print(f"DataFrame columns for @id={record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps: filter numeric fields, normalize values, group/categorize records.

All columns and fields are referenced by their `@id`.

In [ ]:
# Choose numeric field (by @id) from the first record set
# Here we use a placeholder numeric field; update with actual @id from the overview above.
first_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(first_rs_id, pd.DataFrame())

# Find numeric fields
numeric_field_id = None
for rset in getattr(metadata, 'recordSet', []):
    if getattr(rset, '@id', None) == first_rs_id:
        for fld in getattr(rset, 'field', []):
            if getattr(fld, 'dataType', '').lower() in ('float', 'integer', 'number'):
                numeric_field_id = getattr(fld, '@id', None)
                break
        break

if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (e.g., phenotype, or gender, by @id)
    group_field_id = None
    for rset in getattr(metadata, 'recordSet', []):
        if getattr(rset, '@id', None) == first_rs_id:
            for fld in getattr(rset, 'field', []):
                # Find categorical field
                if getattr(fld, 'dataType', '').lower() == 'text':
                    group_field_id = getattr(fld, '@id', None)
                    break
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in first record set.")

## 5. Visualization
Visualize relationships between fields (referenced by `@id`) in the dataset.

In [ ]:
# Visualization example: plot numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.show()

# If grouping field exists, visualize grouped means
if group_field_id and group_field_id in df.columns:
    grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(8,4))
    sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 clinical dataset using `mlcroissant`, referencing all entities by their `@id`. We loaded metadata, reviewed record sets and fields, extracted tabular data, filtered and normalized numeric columns, and visualized relationships.

- All references used dataset schema `@id` fields to ensure reproducibility and clarity.
- This dataset is valuable for rare case studies, though its limited scope restricts broad statistical and AI-driven clinical applications.
- For further research, leverage the Croissant schema to trace provenance, harmonize data, and compare across similar clinical cohorts.